# RoomCraft 3D: tạo model 3D đồ nội thất từ ảnh sản phẩm (Hunyuan3D-2, Kaggle)

**Mỗi thành viên chỉ cần sửa ô ⚙️ CẤU HÌNH, sau đó chạy toàn bộ notebook.**

### Trước khi chạy (làm 1 lần)
1. **Settings → Accelerator → `GPU T4 x2`** (2 GPU, notebook tự chia việc để chạy song song).
2. **Settings → Internet → On** (cần để tải model).
3. **Add Input** → chọn dataset ảnh sản phẩm chung của nhóm (vd. `roomcraft-products`, chính là thư mục `assets/products` của repo). Nếu không có dataset chung, upload ảnh của phần mình là được.

### Cách chạy để không phải ngồi canh
Bấm **Save Version → Save & Run All (Commit)**. Kaggle sẽ chạy nền (tối đa 12 giờ), có thể tắt máy. Khi chạy xong, vào tab **Output** của version đó để tải `roomcraft3d_models.zip`.

### Kết quả
`/kaggle/working/roomcraft3d/models/<product-id>.glb`: đã tách nền, giảm số mặt, có normal mượt, material không kim loại và texture JPEG. Tải zip về, chép các file `.glb` vào thư mục `assets/models/` của repo rồi commit (xem `tools/3d/README.md`).

Nếu bị ngắt giữa chừng, chỉ cần chạy lại: sản phẩm nào đã xong sẽ tự bỏ qua.

In [ ]:
# ⚙️ CẤU HÌNH: mỗi thành viên chỉ sửa ô này

# Danh sách sản phẩm của đợt này (cả nhóm dùng CHUNG một danh sách).
# product-id = tên file ảnh bỏ đuôi "-NN.jpg"
#   vd: product-farmhouse-lawson-swivel-chair-04-01.jpg  ->  product-farmhouse-lawson-swivel-chair-04
# Để trống [] = làm tất cả sản phẩm có trong Input.
PRODUCT_IDS = [
    "product-farmhouse-lawson-swivel-chair-04",
    "product-scandinavian-franco-dining-chair-black-05",
    "product-scandinavian-marble-footed-bowl-white-04",
]

# Chia việc: "<thứ tự của bạn>/<số người>". Bạn thứ 1 điền "1/5", bạn thứ 2 điền "2/5"...
# Làm hết một mình thì để "1/1".
MY_SHARE = "1/1"

# Chất lượng:
#   "fast"     - nhanh nhất, dùng để thử nhanh
#   "balanced" - mặc định, chất lượng tốt cho web
#   "best"     - đẹp nhất (shape 50 bước + texture bản đầy đủ), chậm hơn ~2-3 lần
PRESET = "balanced"

# Số mặt tối đa của model. 40000 đủ đẹp để xem trên web và file nhẹ (~1,5–2 MB).
# Để 0 = dùng mặc định của PRESET (fast 40k, balanced 60k, best 80k).
MAX_FACES = 40000

# Chọn ảnh thủ công cho sản phẩm (mặc định tự chọn ảnh nền trắng đầu tiên).
# Nên chọn ảnh chụp góc 3/4 (thấy cả mặt trước và hông) nếu có, model 3D sẽ đúng chiều sâu hơn.
IMAGE_OVERRIDE = {
    # "product-farmhouse-lawson-swivel-chair-04": 3,   # dùng ảnh ...-03.jpg
}

# Số góc camera khi tô texture: 10 (mặc định, ít lỗ/đốm ở góc khuất) | 6 (bản gốc Hunyuan, nhanh hơn)
# Hết VRAM thì tự giảm 10 -> 8 -> 6.
TEXTURE_VIEWS = 10

# Sản phẩm nào texture bị vẽ sai (mặt sau có hình lạ, màu lệch...) -> thêm vào đây rồi chạy lại từ ô 3.
# Chỉ làm lại bước tô texture (giữ nguyên shape), mỗi seed cho một kết quả khác.
REDO_TEXTURE = {
    # "product-farmhouse-throw-pillow-13-round-pillow-09": 7,   # product-id: seed bất kỳ
}

INPUT_DIR = "/kaggle/input"
OUT_DIR = "/kaggle/working/roomcraft3d"
REMBG_MODEL = "isnet-general-use"   # hoặc "birefnet-general" (đẹp hơn ở chân ghế mảnh, chậm hơn)
TEX_SIZE = 2048

## 1. Cài đặt (~5-8 phút lần đầu)

In [ ]:
import os, sys, subprocess, time
os.chdir("/kaggle/working")
t0 = time.time()

if not os.path.exists("Hunyuan3D-2"):
    subprocess.run("git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git", shell=True, check=True)

# giữ nguyên numpy/torch của Kaggle để không phá môi trường CUDA
import numpy, torch
with open("constraints.txt", "w") as f:
    f.write(f"numpy=={numpy.__version__}\ntorch=={torch.__version__}\n")

subprocess.run(
    f"{sys.executable} -m pip install -q -c constraints.txt "
    "trimesh pygltflib xatlas pymeshlab pybind11 rembg onnxruntime "
    "'transformers==4.57.1' 'huggingface-hub==0.36.0' 'diffusers==0.32.2'",
    shell=True, check=True)

def ext_ok():
    r = subprocess.run([sys.executable, "-c", "import torch, custom_rasterizer, mesh_processor"],
                       capture_output=True, text=True)
    return r.returncode == 0

if not ext_ok():
    for d in ["custom_rasterizer", "differentiable_renderer"]:
        print("Build", d, "...")
        subprocess.run(f"cd /kaggle/working/Hunyuan3D-2/hy3dgen/texgen/{d} && {sys.executable} setup.py install",
                       shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
assert ext_ok(), "Build extension lỗi: kiểm tra đã chọn GPU T4 chưa"

N_GPU = torch.cuda.device_count()
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout)
assert N_GPU > 0, "Chưa bật GPU! Settings → Accelerator → GPU T4 x2"
print(f"✅ Cài đặt xong sau {time.time()-t0:.0f}s, có {N_GPU} GPU")

## 2. Code xử lý (không cần sửa)

In [ ]:
%%writefile /kaggle/working/rc3d_worker.py
# RoomCraft 3D worker — chạy trên 1 GPU, xử lý phần việc (shard) của mình.
# Được notebook gọi tự động; không cần chạy tay.
import argparse
import fcntl
import gc
import io
import json
import os
import sys
import time
import traceback
from contextlib import contextmanager

import numpy as np
from PIL import Image

HY_DIR = "/kaggle/working/Hunyuan3D-2"
if os.path.isdir(HY_DIR):
    sys.path.insert(0, HY_DIR)

PRESETS = {
    # shape_repo, shape_subfolder, steps, octree, flashvdm, paint_subfolder, max_faces
    "fast": dict(shape_repo="tencent/Hunyuan3D-2mini", shape_sub="hunyuan3d-dit-v2-mini-turbo",
                 steps=5, octree=256, flashvdm=True,
                 paint_sub="hunyuan3d-paint-v2-0-turbo", faces=40000),
    "balanced": dict(shape_repo="tencent/Hunyuan3D-2", shape_sub="hunyuan3d-dit-v2-0-turbo",
                     steps=5, octree=384, flashvdm=True,
                     paint_sub="hunyuan3d-paint-v2-0-turbo", faces=60000),
    "best": dict(shape_repo="tencent/Hunyuan3D-2", shape_sub="hunyuan3d-dit-v2-0",
                 steps=50, octree=384, flashvdm=True,
                 paint_sub="hunyuan3d-paint-v2-0", faces=80000),
}


# Góc camera cho bước tô texture. (elev, azim, weight); azim phải là bội số của 30, elev thuộc {0, ±20, ±90}
# vì model Hunyuan-Paint chỉ được train với các góc này.
VIEWS_6 = [(0, 0, 1.0), (0, 90, 0.1), (0, 180, 0.5), (0, 270, 0.1), (90, 0, 0.05), (-90, 180, 0.05)]
# +4 góc chéo nhìn từ trên xuống 20°: thấy được góc trong ghế sofa, khe giữa đệm và lưng ghế...
VIEWS_10 = VIEWS_6 + [(20, 30, 0.5), (20, 330, 0.5), (20, 150, 0.25), (20, 210, 0.25)]
VIEWS_8 = VIEWS_6 + [(20, 30, 0.5), (20, 330, 0.5)]
VIEW_SETS = {10: [VIEWS_10, VIEWS_8, VIEWS_6], 8: [VIEWS_8, VIEWS_6], 6: [VIEWS_6]}


class SeededPipe:
    """Bọc pipeline multiview để đổi được seed (Hunyuan cố định seed = 0)."""
    def __init__(self, pipe):
        self._pipe = pipe
        self.seed = 0

    def __getattr__(self, k):
        return getattr(self._pipe, k)

    def __call__(self, *a, **kw):
        import torch
        kw["generator"] = torch.Generator(device=self._pipe.device).manual_seed(int(self.seed))
        return self._pipe(*a, **kw)


def paint_with_views(paint, mesh, img, views_pref, seed, tag=""):
    """Tô texture với nhiều góc nhìn nhất có thể; hết VRAM thì tự giảm số góc."""
    import torch
    paint.models["multiview_model"].pipeline.seed = seed
    sets = VIEW_SETS.get(views_pref, [VIEWS_6])
    for k, views in enumerate(sets):
        paint.config.candidate_camera_elevs = [v[0] for v in views]
        paint.config.candidate_camera_azims = [v[1] for v in views]
        paint.config.candidate_view_weights = [v[2] for v in views]
        try:
            return paint(mesh.copy(), image=img), len(views)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if k == len(sets) - 1:
                raise
            log(f"hết VRAM với {len(views)} góc, thử {len(sets[k + 1])} góc", tag)


def log(msg, tag=""):
    print(f"[{time.strftime('%H:%M:%S')}]{tag} {msg}", flush=True)


@contextmanager
def load_lock(path="/kaggle/working/.model_load.lock"):
    """Chỉ cho 1 worker load model tại một thời điểm -> tránh tràn RAM khi chạy 2 GPU."""
    with open(path, "w") as f:
        fcntl.flock(f, fcntl.LOCK_EX)
        try:
            yield
        finally:
            fcntl.flock(f, fcntl.LOCK_UN)


# ----------------------------------------------------------------------------
# 1. Tách nền
# ----------------------------------------------------------------------------
class BgRemover:
    def __init__(self, model_name="isnet-general-use"):
        from rembg import new_session
        try:
            self.session = new_session(model_name)
            self.name = model_name
        except Exception as e:
            log(f"Không load được {model_name} ({e}) -> dùng u2net")
            self.session = new_session("u2net")
            self.name = "u2net"

    def __call__(self, img: Image.Image) -> Image.Image:
        from rembg import remove
        rgba = remove(img.convert("RGB"), session=self.session, post_process_mask=True)
        a = np.array(rgba)
        # dọn alpha: bỏ viền mờ / bóng đổ nhạt dưới chân sản phẩm
        alpha = a[..., 3]
        alpha[alpha < 40] = 0
        a[..., 3] = alpha
        return Image.fromarray(a, "RGBA")


def square_pad(rgba: Image.Image, border=0.12, size=1024) -> Image.Image:
    """Cắt sát vật thể, thêm lề, đưa về ảnh vuông -> model 3D hiểu tỉ lệ tốt hơn."""
    a = np.array(rgba)
    ys, xs = np.nonzero(a[..., 3] > 0)
    if len(xs) == 0:
        raise ValueError("Tách nền thất bại: ảnh trống")
    crop = rgba.crop((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))
    w, h = crop.size
    side = int(max(w, h) * (1 + 2 * border))
    canvas = Image.new("RGBA", (side, side), (255, 255, 255, 0))
    canvas.paste(crop, ((side - w) // 2, (side - h) // 2))
    return canvas.resize((size, size), Image.LANCZOS)


# ----------------------------------------------------------------------------
# 2. Hậu xử lý cho web
# ----------------------------------------------------------------------------
def _append_image_hq(img, tree, buffer_items, extension_webp):
    """Thay hàm của trimesh để ghi texture JPEG chất lượng 90 (mặc định trimesh ghi PNG rất nặng)."""
    from trimesh.exchange import gltf as _g
    if not hasattr(img, "save"):
        return None
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format="JPEG", quality=90, subsampling=0)
    index = _g._buffer_append(buffer_items, buf.getvalue())
    tree["images"].append({"bufferView": index, "mimeType": "image/jpeg"})
    return len(tree["images"]) - 1


def finalize_for_web(mesh, out_path, tex_size=2048):
    """mesh: trimesh.Trimesh có TextureVisuals (output của Hunyuan3DPaintPipeline)."""
    import trimesh
    from trimesh.exchange import gltf as _g
    from trimesh.visual.material import PBRMaterial

    if isinstance(mesh, trimesh.Scene):
        mesh = mesh.to_geometry()
    v = np.asarray(mesh.vertices, dtype=np.float64)
    f = np.asarray(mesh.faces, dtype=np.int64)
    uv = np.asarray(mesh.visual.uv, dtype=np.float64)
    mat = mesh.visual.material
    tex = getattr(mat, "image", None) or getattr(mat, "baseColorTexture", None)

    # Normal mượt: hàn các đỉnh trùng vị trí (bị tách do đường may UV) rồi tính normal
    _, first, inv = np.unique(np.round(v, 5), axis=0, return_index=True, return_inverse=True)
    inv = np.asarray(inv).reshape(-1)
    welded = trimesh.Trimesh(vertices=v[first], faces=inv[f], process=False)
    normals = np.asarray(welded.vertex_normals)[inv]

    # Đặt vật lên mặt sàn (y = 0), căn giữa x/z
    lo, hi = v.min(0), v.max(0)
    v = v - np.array([(lo[0] + hi[0]) / 2, lo[1], (lo[2] + hi[2]) / 2])

    if tex is not None and max(tex.size) > tex_size:
        tex = tex.resize((tex_size, tex_size), Image.LANCZOS)
    material = PBRMaterial(
        name="roomcraft",
        baseColorTexture=tex,
        baseColorFactor=[255, 255, 255, 255],
        metallicFactor=0.0,      # quan trọng: mặc định glTF = 1.0 -> model tối như kim loại
        roughnessFactor=0.8,
        doubleSided=False,
    )
    out = trimesh.Trimesh(
        vertices=v, faces=f, vertex_normals=normals,
        visual=trimesh.visual.TextureVisuals(uv=uv, material=material),
        process=False,
    )
    orig = _g._append_image
    _g._append_image = _append_image_hq
    tmp = out_path + ".part"
    try:
        data = out.export(file_type="glb", include_normals=True)
        with open(tmp, "wb") as fh:
            fh.write(data)
        os.replace(tmp, out_path)   # chỉ xuất hiện file .glb khi đã ghi xong -> chạy lại không bị bỏ sót
    finally:
        _g._append_image = orig
        if os.path.exists(tmp):
            os.remove(tmp)
    return out_path


# ----------------------------------------------------------------------------
# 3. Pipeline chính
# ----------------------------------------------------------------------------
STAGE = {"name": "khởi động"}
STATE = {}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--jobs", required=True)
    ap.add_argument("--worker", type=int, default=0)
    ap.add_argument("--num-workers", type=int, default=1)
    ap.add_argument("--out", default="/kaggle/working/roomcraft3d")
    ap.add_argument("--preset", default="balanced", choices=list(PRESETS))
    ap.add_argument("--rembg", default="isnet-general-use")
    ap.add_argument("--tex-size", type=int, default=2048)
    ap.add_argument("--seed", type=int, default=1234)
    ap.add_argument("--views", type=int, default=10, choices=[6, 8, 10])
    ap.add_argument("--max-faces", type=int, default=0)
    args = ap.parse_args()
    try:
        _run(args)
    except BaseException as e:
        # lỗi ngoài vòng lặp từng sản phẩm (load model, thiếu thư viện, hết RAM...) -> ghi lại cho ô tổng kết
        tb = traceback.format_exc()
        print(tb, flush=True)
        msg = f"{type(e).__name__}: {e}"[:400]
        for j in STATE.get("todo", []):
            if not STATE["is_done"](STATE["final_path"](j)):
                STATE["record"](j, ok=False, stage=STAGE["name"], error=msg)
        sys.exit(1)


def _run(args):
    STAGE["name"] = "import thư viện"
    import torch
    import trimesh

    P = PRESETS[args.preset]
    tag = f"[GPU{args.worker}]"
    jobs = json.load(open(args.jobs))[args.worker::args.num_workers]
    dirs = {k: os.path.join(args.out, k) for k in ("models", "shapes", "inputs", "logs")}
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)
    status_path = os.path.join(dirs["logs"], f"status_worker{args.worker}.jsonl")

    def record(job, **kw):
        with open(status_path, "a") as fh:
            fh.write(json.dumps({"id": job["id"], "image": os.path.basename(job["image"]), **kw}) + "\n")

    def final_path(j):
        return os.path.join(dirs["models"], f"{j['id']}.glb")

    def is_done(path):
        return os.path.exists(path) and os.path.getsize(path) > 1024

    todo = [j for j in jobs if not is_done(final_path(j))]
    STATE.update(todo=todo, is_done=is_done, final_path=final_path, record=record)
    log(f"{len(jobs)} sản phẩm được giao, {len(jobs) - len(todo)} đã xong trước đó, còn {len(todo)}", tag)
    if not todo:
        return

    timing = {j["id"]: {} for j in todo}

    # ---------- Bước A: tách nền ----------
    STAGE["name"] = "tách nền"
    t = time.time()
    rem = BgRemover(args.rembg)
    for j in todo:
        p = os.path.join(dirs["inputs"], f"{j['id']}.png")
        if os.path.exists(p):
            continue
        t1 = time.time()
        try:
            square_pad(rem(Image.open(j["image"]))).save(p)
            timing[j["id"]]["rembg"] = round(time.time() - t1, 1)
        except Exception as e:
            log(f"✗ {j['id']} tách nền lỗi: {e}", tag)
            record(j, ok=False, stage="rembg", error=str(e))
    del rem
    gc.collect()
    log(f"Tách nền xong ({time.time() - t:.0f}s, model {args.rembg})", tag)

    # ---------- Bước B: tạo hình khối ----------
    STAGE["name"] = "load shape model"
    from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
    from hy3dgen.shapegen.postprocessors import FloaterRemover, DegenerateFaceRemover, FaceReducer

    need_shape = [j for j in todo
                  if os.path.exists(os.path.join(dirs["inputs"], f"{j['id']}.png"))
                  and not is_done(os.path.join(dirs["shapes"], f"{j['id']}.glb"))]
    if need_shape:
        with load_lock():
            log(f"Load shape model {P['shape_sub']} ...", tag)
            shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
                P["shape_repo"], subfolder=P["shape_sub"], variant="fp16", use_safetensors=True)
            if P["flashvdm"]:
                shape_pipe.enable_flashvdm(mc_algo="mc")
        floater, degen, reducer = FloaterRemover(), DegenerateFaceRemover(), FaceReducer()
        STAGE["name"] = "tạo shape"

        for i, j in enumerate(need_shape, 1):
            t1 = time.time()
            try:
                img = Image.open(os.path.join(dirs["inputs"], f"{j['id']}.png"))
                mesh = shape_pipe(
                    image=img,
                    num_inference_steps=P["steps"],
                    octree_resolution=P["octree"],
                    num_chunks=20000,
                    generator=torch.manual_seed(args.seed),
                    output_type="trimesh",
                )[0]
                mesh = floater(mesh)
                mesh = degen(mesh)
                mesh = reducer(mesh, max_facenum=(args.max_faces or P["faces"]))   # <-- chìa khoá tăng tốc bước texture
                mesh.export(os.path.join(dirs["shapes"], f"{j['id']}.glb"))
                timing[j["id"]]["shape"] = round(time.time() - t1, 1)
                log(f"shape {i}/{len(need_shape)} {j['id']}: {len(mesh.faces)} mặt, {time.time() - t1:.0f}s", tag)
            except Exception as e:
                log(f"✗ {j['id']} shape lỗi: {e}", tag)
                traceback.print_exc()
                record(j, ok=False, stage="shape", error=str(e))
            torch.cuda.empty_cache()
        del shape_pipe
        gc.collect()
        torch.cuda.empty_cache()

    # ---------- Bước C: tô texture ----------
    need_tex = [j for j in todo if is_done(os.path.join(dirs["shapes"], f"{j['id']}.glb"))]
    if not need_tex:
        return
    STAGE["name"] = "load texture model"
    from hy3dgen.texgen import Hunyuan3DPaintPipeline

    with load_lock():
        log(f"Load texture model {P['paint_sub']} ...", tag)
        paint = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2", subfolder=P["paint_sub"])
    mvm = paint.models["multiview_model"]
    mvm.pipeline = SeededPipe(mvm.pipeline)
    log(f"VRAM sau khi load: {torch.cuda.memory_allocated() / 1024**3:.1f} GB", tag)
    STAGE["name"] = "tô texture"

    for i, j in enumerate(need_tex, 1):
        t1 = time.time()
        try:
            mesh = trimesh.load(os.path.join(dirs["shapes"], f"{j['id']}.glb"), force="mesh")
            img = Image.open(os.path.join(dirs["inputs"], f"{j['id']}.png"))
            seed = j.get("tex_seed", 0)
            textured, n_views = paint_with_views(paint, mesh, img, args.views, seed, tag)
            j["views_used"] = n_views
            finalize_for_web(textured, final_path(j), tex_size=args.tex_size)
            timing[j["id"]]["texture"] = round(time.time() - t1, 1)
            mb = os.path.getsize(final_path(j)) / 1024 / 1024
            total = sum(timing[j["id"]].values())
            log(f"✓ texture {i}/{len(need_tex)} {j['id']}: {mb:.1f} MB, {time.time() - t1:.0f}s "
                f"(tổng ~{total / 60:.1f} phút)", tag)
            record(j, ok=True, mb=round(mb, 2), seconds=timing[j["id"]], preset=args.preset,
                   views=n_views, tex_seed=seed)
        except Exception as e:
            log(f"✗ {j['id']} texture lỗi: {e}", tag)
            traceback.print_exc()
            record(j, ok=False, stage="texture", error=str(e))
        gc.collect()
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/rc3d_select.py
import os, re
import numpy as np
from PIL import Image
from collections import defaultdict

IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp")
NAME_RE = re.compile(r"^(?P<pid>.+)-(?P<idx>\d{1,3})$")

def studio_score(path):
    """Tỉ lệ pixel viền ảnh là nền trắng/xám nhạt (ảnh studio ~1.0, ảnh chụp trong phòng thấp)."""
    im = np.asarray(Image.open(path).convert("RGB").resize((256, 256)), dtype=np.int16)
    b = 8
    border = np.concatenate([im[:b].reshape(-1, 3), im[-b:].reshape(-1, 3),
                             im[:, :b].reshape(-1, 3), im[:, -b:].reshape(-1, 3)])
    bright = border.min(1) > 215
    flat = (border.max(1) - border.min(1)) < 18
    return float((bright & flat).mean())

def build_jobs(input_dir, product_ids=None, overrides=None, min_score=0.85):
    groups = defaultdict(list)
    for root, _, files in os.walk(input_dir):
        for f in files:
            stem, ext = os.path.splitext(f)
            if ext.lower() not in IMG_EXTS:
                continue
            m = NAME_RE.match(stem)
            pid, idx = (m["pid"], int(m["idx"])) if m else (stem, 0)
            groups[pid].append((idx, os.path.join(root, f)))
    wanted = set(product_ids) if product_ids else set(groups)
    missing = sorted(wanted - set(groups))
    jobs, warns = [], []
    for pid in sorted(wanted & set(groups)):
        imgs = sorted(groups[pid])
        chosen = None
        if overrides and pid in overrides:
            chosen = next((p for i, p in imgs if i == int(overrides[pid])), None)
        if chosen is None:
            scored = [(studio_score(p), i, p) for i, p in imgs]
            clean = [s for s in scored if s[0] >= min_score]
            best = clean[0] if clean else max(scored)
            chosen = best[2]
            if not clean:
                warns.append(f"{pid}: không có ảnh nền trắng rõ ràng, dùng {os.path.basename(chosen)} (score {best[0]:.2f})")
        jobs.append({"id": pid, "image": chosen})
    return jobs, missing, warns

## 3. Chọn ảnh cho từng sản phẩm

In [ ]:
import json, sys, os
sys.path.insert(0, "/kaggle/working")
import importlib, rc3d_select
importlib.reload(rc3d_select)

all_jobs, missing, warns = rc3d_select.build_jobs(INPUT_DIR, PRODUCT_IDS or None, IMAGE_OVERRIDE)

k, n = map(int, MY_SHARE.split("/"))
jobs = all_jobs[k - 1::n]

for j in jobs:
    if j["id"] in REDO_TEXTURE:
        j["tex_seed"] = int(REDO_TEXTURE[j["id"]])
        old_glb = f"{OUT_DIR}/models/{j['id']}.glb"
        if os.path.exists(old_glb):
            os.remove(old_glb)   # xoá bản cũ -> worker sẽ tô lại texture (shape giữ nguyên)
            print(f"↻ Tô lại texture {j['id']} với seed {j['tex_seed']}")

os.makedirs(OUT_DIR, exist_ok=True)
json.dump(jobs, open("/kaggle/working/jobs.json", "w"), indent=1)

print(f"Tổng {len(all_jobs)} sản phẩm, phần của bạn ({MY_SHARE}): {len(jobs)}")
for j in jobs:
    print(f"  • {j['id']}  ←  {os.path.basename(j['image'])}")
for w in warns:
    print("⚠️", w)
if missing:
    print(f"⚠️ Không tìm thấy ảnh cho {len(missing)} sản phẩm:", *missing[:20], sep="\n   ")
assert jobs, "Không có sản phẩm nào để chạy, kiểm tra PRODUCT_IDS / Input"

## 4. Tải model (chỉ tải đúng file cần, bỏ file trùng .ckpt/.safetensors không dùng tới)

In [ ]:
import huggingface_hub as hf
sys.path.insert(0, "/kaggle/working")
from rc3d_worker import PRESETS
P = PRESETS[PRESET]
CACHE = os.path.expanduser("~/.cache/hy3dgen")

def get(repo, patterns, ignore=None):
    hf.snapshot_download(repo_id=repo, allow_patterns=patterns, ignore_patterns=ignore,
                         local_dir=os.path.join(CACHE, repo))

t0 = time.time()
get(P["shape_repo"], [f"{P['shape_sub']}/config.yaml", f"{P['shape_sub']}/model.fp16.safetensors"])
if P["flashvdm"]:
    vae_repo, vae_sub = (("tencent/Hunyuan3D-2mini", "hunyuan3d-vae-v2-mini-turbo")
                         if "mini" in P["shape_repo"] else ("tencent/Hunyuan3D-2", "hunyuan3d-vae-v2-0-turbo"))
    get(vae_repo, [f"{vae_sub}/config.yaml", f"{vae_sub}/model.fp16.safetensors"])
get("tencent/Hunyuan3D-2", ["hunyuan3d-delight-v2-0/*", f"{P['paint_sub']}/*"],
    # Lưu ý: unet của paint được Hunyuan đọc CỨNG từ file .bin -> bỏ bản .safetensors (không dùng tới)
    ignore=["hunyuan3d-paint-*/unet/diffusion_pytorch_model.safetensors"])
unet_bin = os.path.join(CACHE, "tencent/Hunyuan3D-2", P["paint_sub"], "unet/diffusion_pytorch_model.bin")
assert os.path.exists(unet_bin), f"Thiếu {unet_bin}, chạy lại ô này"
print(f"✅ Model sẵn sàng ({time.time()-t0:.0f}s)")

## 5. Chạy: tách nền → tạo hình khối → tô texture → xuất .glb cho web

In [ ]:
import subprocess, time, glob

n_workers = min(N_GPU, len(jobs))
logs, procs = [], []
for g in range(n_workers):
    log_path = f"{OUT_DIR}/logs/run_gpu{g}.log"
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    fh = open(log_path, "w")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PYTHONUNBUFFERED="1")
    procs.append(subprocess.Popen(
        [sys.executable, "/kaggle/working/rc3d_worker.py", "--jobs", "/kaggle/working/jobs.json",
         "--worker", str(g), "--num-workers", str(n_workers), "--out", OUT_DIR,
         "--preset", PRESET, "--rembg", REMBG_MODEL, "--tex-size", str(TEX_SIZE), "--max-faces", str(MAX_FACES),
         "--views", str(TEXTURE_VIEWS)],
        stdout=fh, stderr=subprocess.STDOUT, env=env, cwd="/kaggle/working/Hunyuan3D-2"))
    logs.append([log_path, 0])
print(f"Đang chạy {n_workers} worker song song cho {len(jobs)} sản phẩm (preset {PRESET})...\n")

# in tiến độ (lọc bớt thanh progress của diffusers)
t0 = time.time()
def pump():
    for item in logs:
        with open(item[0], errors="ignore") as f:
            f.seek(item[1]); data = f.read(); item[1] = f.tell()
        for line in data.replace("\r", "\n").splitlines():
            if line.startswith("[") and ("GPU" in line[:20]) or "Error" in line or "Traceback" in line:
                print(line)
while any(p.poll() is None for p in procs):
    pump(); time.sleep(15)
pump()
print(f"\n⏱️ Tổng thời gian: {(time.time()-t0)/60:.1f} phút")
def show_tail(path, n=40):
    lines = [l for l in open(path, errors="ignore").read().replace("\r", "\n").splitlines()
             if l.strip() and "it/s]" not in l and "s/it]" not in l]
    print("\n".join(lines[-n:]))

for g, p in enumerate(procs):
    if p.returncode != 0:
        why = " → bị hệ thống kill, gần như chắc chắn do HẾT RAM" if p.returncode in (-9, 137) else ""
        print(f"\n❌ Worker GPU{g} thoát lỗi (code {p.returncode}){why}. 40 dòng cuối log:\n" + "-" * 70)
        show_tail(logs[g][0])
        print("-" * 70)

## 6. Kiểm tra kết quả & nén để tải về

In [ ]:
import json, glob, zipfile, os
from PIL import Image
import matplotlib.pyplot as plt

done = {os.path.basename(p)[:-4]: p for p in glob.glob(f"{OUT_DIR}/models/*.glb")}
status = {}
for p in glob.glob(f"{OUT_DIR}/logs/status_worker*.jsonl"):
    for line in open(p):
        r = json.loads(line); status[r["id"]] = r

print(f"✅ Xong {sum(j['id'] in done for j in jobs)}/{len(jobs)} sản phẩm\n")
for j in jobs:
    s = status.get(j["id"], {})
    if j["id"] in done:
        sec = s.get("seconds", {})
        extra = f", {s['views']} góc, seed {s.get('tex_seed', 0)}" if "views" in s else ""
        print(f"  ✓ {j['id']}: {os.path.getsize(done[j['id']])/1e6:.1f} MB, "
              f"{sum(sec.values())/60:.1f} phút ({', '.join(f'{k} {v:.0f}s' for k, v in sec.items())}{extra})")
    else:
        print(f"  ✗ {j['id']}: lỗi ở bước {s.get('stage','?')}: {s.get('error','worker bị dừng đột ngột (thường do hết RAM), xem log bên dưới')}")

if any(j["id"] not in done for j in jobs):
    for lp in sorted(glob.glob(f"{OUT_DIR}/logs/run_gpu*.log")):
        lines = [l for l in open(lp, errors="ignore").read().replace("\r", "\n").splitlines()
                 if l.strip() and "it/s]" not in l and "s/it]" not in l]
        print(f"\n----- 30 dòng cuối {os.path.basename(lp)} -----")
        print("\n".join(lines[-30:]))

# Xem lại ảnh đã tách nền: nếu ảnh nào bị cắt sai, đổi ảnh bằng IMAGE_OVERRIDE rồi chạy lại
ins = [j for j in jobs if os.path.exists(f"{OUT_DIR}/inputs/{j['id']}.png")]
if ins:
    cols = min(6, len(ins)); rows = (len(ins) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3.2 * rows), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, j in zip(axes.flat, ins):
        im = Image.open(f"{OUT_DIR}/inputs/{j['id']}.png")
        bg = Image.new("RGBA", im.size, (235, 90, 90, 255)); bg.alpha_composite(im)
        ax.imshow(bg); ax.set_title(("✓ " if j["id"] in done else "✗ ") + j["id"][-32:], fontsize=8)
    plt.suptitle("Ảnh đầu vào sau khi tách nền (nền đỏ = phần bị loại bỏ)"); plt.tight_layout(); plt.show()

zip_path = "/kaggle/working/roomcraft3d_models.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
    for p in done.values():
        z.write(p, f"models/{os.path.basename(p)}")
    for p in glob.glob(f"{OUT_DIR}/logs/*"):
        z.write(p, f"logs/{os.path.basename(p)}")
print(f"\n📦 {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB): tải ở tab Output")

## Khi kết quả chưa đẹp
| Hiện tượng | Cách xử lý |
|---|---|
| Có mảng sàn/bóng dính vào model | Xem ảnh tách nền ở ô 6; đổi `REMBG_MODEL = "birefnet-general"` hoặc chọn ảnh khác bằng `IMAGE_OVERRIDE` |
| Model bị dẹt / sai chiều sâu | Dùng ảnh góc 3/4 thay vì ảnh chính diện (`IMAGE_OVERRIDE`) |
| Chân ghế/bàn mảnh bị đứt | `PRESET = "best"` và `REMBG_MODEL = "birefnet-general"` |
| Texture có lỗ/đốm ở góc khuất (khe đệm sofa, chân ghế) | Giữ `TEXTURE_VIEWS = 10` (thêm 4 góc chéo nhìn từ trên) |
| Mặt sau/mặt bên có hình lạ, chữ lạ | AI phải tự đoán phần không có trong ảnh. Thêm SP vào `REDO_TEXTURE` với seed khác (vd 7, 21, 42) rồi chạy lại từ ô 3; chọn ảnh góc 3/4 cũng giúp nhiều |
| Lỗi CUDA out of memory | Chuyển `PRESET = "fast"`, hoặc kiểm tra đã chọn T4 x2 (không phải P100) |

Muốn chạy lại 1 sản phẩm: xoá `roomcraft3d/models/<id>.glb`, `shapes/<id>.glb` và `inputs/<id>.png` rồi chạy lại ô 5.